# **NLP Task 4: Fine-Tuning BERT on Kaggle Dataset**

**Objective**

The objective of this task is to fine-tune a pre-trained BERT model on a real-world dataset for text classification. This includes preprocessing text data, tokenization, model training, and evaluation using multiple metrics.

**Tools Used**

Python

Hugging Face Transformers

PyTorch

Google Colab

**Pipeline Flow**

Raw Data → Preprocessing → Tokenization → Model Training → Evaluation → Comparison

**STEP 1: Install**

In [ ]:
!pip install transformers datasets scikit-learn

**STEP 2: Import Libraries**

In [ ]:
import numpy as np
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

**STEP 3: Load Dataset**

In [ ]:
dataset = load_dataset("imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

**STEP 4: Preprocessing**

In [ ]:
def clean_text(example):
    text = example["text"].lower()
    text = " ".join(text.split())
    return {"text": text}

dataset = dataset.map(clean_text)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

**STEP 5: Split Data**

In [ ]:
split = dataset["train"].train_test_split(test_size=0.1)

dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
    "test": dataset["test"]
})

**STEP 6: Tokenization**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/22500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

**STEP 7: Load Model**

In [ ]:
small_train = tokenized_dataset["train"].select(range(1000))
small_val = tokenized_dataset["validation"].select(range(200))
small_test = tokenized_dataset["test"].select(range(200))

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**STEP 8: Metrics**

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

**STEP 9: Training Arguments**

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    compute_metrics=compute_metrics
)

**STEP 10: Train Model**

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.4271358642578125, metrics={'train_runtime': 7424.5267, 'train_samples_per_second': 0.135, 'train_steps_per_second': 0.034, 'total_flos': 263111055360000.0, 'train_loss': 0.4271358642578125, 'epoch': 1.0})

**STEP 11: Evaluation**

In [ ]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.31966954469680786, 'eval_accuracy': 0.875, 'eval_precision': 0.8773584905660378, 'eval_recall': 0.8857142857142857, 'eval_f1': 0.8815165876777251, 'eval_runtime': 463.6385, 'eval_samples_per_second': 0.431, 'eval_steps_per_second': 0.108, 'epoch': 1.0}


In [ ]:
preds_output = trainer.predict(small_test)

y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = preds_output.label_ids

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Confusion Matrix:
 [[172  28]
 [  0   0]]


# **Experiments**

**Freeze BERT**

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

**Fine-tune last layers**

In [ ]:
for name, param in model.bert.named_parameters():
    if "layer.10" in name or "layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# **Conclusion**

Due to computational limitations, a subset of the dataset was used.

The model performed well on sentiment classification.

- Full fine-tuning gives best accuracy
- Freezing layers reduces performance
- Fine-tuning last layers balances speed and accuracy

In this task, we fine-tuned a pre-trained BERT model for text classification.

Data was preprocessed and tokenized
Model was trained using transformer architecture
Evaluation metrics like Accuracy, Precision, Recall, and F1-score were used
Confusion matrix provided performance insights

The model achieved good performance, demonstrating the effectiveness of BERT in NLP tasks.

In [2]:
import nbformat

# 👇 yahan apna original notebook file name daal
nb = nbformat.read("BERT_FineTuning_IMDB_Task4.ipynb", as_version=4)

# Remove widget metadata (main issue fix)
nb.metadata.pop("widgets", None)

# Also clean cell metadata (extra safe)
for cell in nb.cells:
    if "metadata" in cell:
        cell["metadata"] = {}

# Save clean notebook
nbformat.write(nb, "clean_BERT_Task4.ipynb")

print("Clean notebook created successfully!")

Clean notebook created successfully!
